# Qwen3.5 Image-to-SMT via Grammar-Constrained Decoding


> Originally, adapted from [Qwen3_5_(0_8B)_Vision.ipynb](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(0_8B)_Vision.ipynb#scrollTo=gGFzmplrEy9I)

![Qwen3.5](https://qianwen-res.oss-accelerate.aliyuncs.com/logo_qwen3.5.png)

Qwen3.5 features the following enhancement:

- **Unified Vision-Language Foundation**: Early fusion training on multimodal tokens achieves cross-generational parity with Qwen3 and outperforms Qwen3-VL models across reasoning, coding, agents, and visual understanding benchmarks.
- **Efficient Hybrid Architecture**: Gated Delta Networks combined with sparse Mixture-of-Experts deliver high-throughput inference with minimal latency and cost overhead.
- **Scalable RL Generalization**: Reinforcement learning scaled across million-agent environments with progressively complex task distributions for robust real-world adaptability.
- **Global Linguistic Coverage**: Expanded support to 201 languages and dialects, enabling inclusive, worldwide deployment with nuanced cultural and regional understanding.
- **Next-Generation Training Infrastructure**: Near-100% multimodal training efficiency compared to text-only training and asynchronous RL frameworks supporting massive-scale agent scaffolds and environment orchestration.

In [ ]:
import os
import subprocess
import tempfile
import json
from pathlib import Path
from outlines.types import CFG
from PIL import Image
from tqdm.auto import tqdm
from collections import defaultdict
from outlines.inputs import Chat
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

import outlines

In [ ]:
MODEL_ID = ""
MAX_NEW_TOKENS = 256

# https://unsloth.ai/docs/models/qwen3-how-to-run-and-fine-tune#official-recommended-settings
ENABLE_THINKING = False
TEMPERATURE = 0.7
MIN_P = 0.01
TOP_P = 0.8
TOP_K = 20

BASE_DIR = Path.cwd().parent
CATEGORY = "train"

COMPETITION_DATA_DIR = BASE_DIR / "ALD-E-ImageMiner" / "icdar2026-competition-data"
CASE_DIR = COMPETITION_DATA_DIR / CATEGORY

STATE_FILE = BASE_DIR / "smt_state.json"
SMT_FILE = BASE_DIR / "smt.json"

CVC5_PATH = Path.home() / "cvc5-Linux-x86_64-shared" / "bin" / "cvc5"

In [ ]:
model = Qwen2VLForConditionalGeneration.from_pretrained(MODEL_ID, device_map="auto")
processor = AutoProcessor.from_pretrained(MODEL_ID)

llm = outlines.from_transformers(model, processor.tokenizer)

In [ ]:
# A stripped down EBNF conforming to our rules

SMT_LIB_GRAMMAR = r"""
?start: script

script: command+
command: "(" "declare-sort" SYMBOL "0" ")"
       | "(" "declare-fun" SYMBOL "(" sort* ")" sort ")"
       | "(" "declare-const" SYMBOL sort ")"
       | "(" "assert" term ")"
       | "(" "check-sat" ")"
       | "(" "get-value" "(" term+ ")" ")"

?term: spec_constant
     | SYMBOL
     | "(" SYMBOL term+ ")"
     | "(" RESERVED_OP term+ ")"

spec_constant: NUMERAL | DECIMAL | STRING_LIT

sort: "Real" | "Bool" | "String" | "Int" | SYMBOL

RESERVED_OP: "=" | "distinct" | "not" | "and" | "or" | "xor" | "=>" | "ite"
           | "<" | "<=" | ">" | ">=" | "+" | "-" | "*" | "/"

SYMBOL: /[a-zA-Z\+\-\*\/\?\!\$\^\_\<\>\=\~][a-zA-Z0-9\+\-\*\/\?\!\$\^\_\<\>\=\~]*/
NUMERAL: /[0-9]+/
DECIMAL: /[0-9]+\.[0-9]+/
STRING_LIT: /"([^"\n]|"")*"/

%import common.WS
%ignore WS
"""

# Compile the grammar
SMT_CFG = CFG(SMT_LIB_GRAMMAR)
generator = outlines.Generator(llm, SMT_CFG)

<a name="Data"></a>
### 🧪 Data Preparation

To convert our Sci-ImageMiner VQA data into the format required by Qwen3.5 (specifically for use with Unsloth), we need to restructure the data into a "messages" format.

The Qwen/Unsloth format expects a list of conversations where the image and the text prompt are bundled together in the user role, and the ground truth is in the assistant role, as follows:

```python
[
    { "role": "user",
    "content": [{"type": "text",  "text": Q}, {"type": "image", "image": image} ]
    },
    { "role": "assistant",
    "content": [{"type": "text",  "text": A} ]
    },
]
```

In [ ]:
PROMPT_YES_NO = """
<image>

[PAPER CONTEXT]
{context}

[METADATA]
Question Type: {question_type}
Answer Type: Yes/No
Question: {question}

[TASK]
You are a formal logic extractor. Convert the visual evidence and context into a discrete SMT-LIB 2.6 reasoning program. Reason ONLY from explicitly extracted visual anchors.

[STRICT RULES]
- NO QUANTIFIERS: Do NOT use 'forall' or 'exists'.
- DISCRETE ANCHORS: Numeric reasoning must rely on extracted coordinates.
- DOMAIN BOUNDS: Must stay within visible axis ranges.
- LEGEND/PANEL GROUNDING: Every series must be mapped to a material and panel.
- FINITE SEARCH: Use (or ...) for maxima/minima over extracted x-values.

[SMT-LIB SCHEMA]
(declare-sort Series 0) (declare-sort Material 0) (declare-sort Panel 0)
(declare-fun material_name (Material) String)
(declare-fun series_material (Series Material) Bool)
(declare-fun val (Series Real) Real)
(declare-const AnsBool Bool)

[LOGIC EXTRACTION]
;; Example extraction for a Yes/No question:
(declare-const s1 Series) (declare-const m1 Material)
(assert (= (material_name m1) "Al2O3"))
(assert (series_material s1 m1))
(assert (= (val s1 250.0) 1.2)) ;; Anchor from figure/context
(assert (= AnsBool (> (val s1 250.0) 1.0)))

(check-sat)
(get-value (AnsBool))
"""

In [ ]:
PROMPT_FACTOID = """
<image>

[PAPER CONTEXT]
{context}

[METADATA]
Question Type: {question_type}
Answer Type: Factoid
Question: {question}

[TASK]
;; (Same Task and Strict Rules as Yes/No)

[SMT-LIB SCHEMA]
;; (Standard sorts + AnsString or AnsReal)
(declare-const AnsString String)
(declare-const AnsReal Real)

[LOGIC EXTRACTION]
;; Example extraction for identifying a material:
(declare-const s1 Series) (declare-const m1 Material) (assert (= (material_name m1) "HfO2"))
(declare-const s2 Series) (assert (= (material_name m2) "ZrO2"))
(assert (= (val s1 300.0) 0.8))
(assert (= (val s2 300.0) 1.5))

;; Logic: Which is higher?
(assert (ite (> (val s2 300.0) (val s1 300.0))
             (= AnsString "ZrO2")
             (= AnsString "HfO2")))

(check-sat)
(get-value (AnsString))
"""

In [ ]:
PROMPT_LIST = """
<image>

[PAPER CONTEXT]
{context}

[METADATA]
Question Type: {question_type}
Answer Type: List
Question: {question}

[TASK]
;; (Same Task and Strict Rules as Yes/No)

[LOGIC EXTRACTION]
;; Define one boolean flag per material found in legend/context.
(declare-const s1 Series) (declare-const m1 Material) (assert (= (material_name m1) "Material_A"))
(declare-const s2 Series) (declare-const m2 Material) (assert (= (material_name m2) "Material_B"))
(assert (= (val s1 400.0) 2.2))
(assert (= (val s2 400.0) 0.5))

;; List Membership Logic
(declare-const in_list_m1 Bool)
(declare-const in_list_m2 Bool)
(assert (= in_list_m1 (> (val s1 400.0) 1.0)))
(assert (= in_list_m2 (> (val s2 400.0) 1.0)))

(check-sat)
(get-value (in_list_m1 in_list_m2))
"""

In [ ]:
PROMPT_PARAGRAPH = """
<image>

[PAPER CONTEXT]
{context}

[METADATA]
Question Type: {question_type}
Answer Type: Paragraph
Question: {question}

[TASK]
;; (Same Task and Strict Rules as Yes/No)

[LOGIC EXTRACTION]
;; Solve for key comparative facts (deltas, slopes, peaks).
(declare-const s1 Series)
(assert (= (val s1 100.0) 0.4))
(assert (= (val s1 500.0) 2.4))

;; Reasoning anchors for the final 3-sentence explanation
(declare-const total_growth Real)
(declare-const growth_is_linear Bool)
(assert (= total_growth (- (val s1 500.0) (val s1 100.0))))
(assert (= growth_is_linear true)) ;; Based on visual trend

(check-sat)
(get-value (total_growth growth_is_linear))
"""

In [ ]:
PROMPTS = {
    "Yes/No": PROMPT_YES_NO,
    "Factoid": PROMPT_FACTOID,
    "List": PROMPT_LIST,
    "Paragraph": PROMPT_PARAGRAPH,
}

In [ ]:
def get_paper_context(json_file_path, window_size=2):
    """
    Finds the parent content.json, extracts the image caption, and
    grabs a sliding window of text blocks (e.g., 2 before, 2 after)
    surrounding the image for highly targeted context.
    """
    # Navigate up from .../16/images/fig_2.json to .../16/content.json
    content_json_path = json_file_path.parent.parent / "content.json"

    assert content_json_path.exists(), f"{content_json_path}"

    # The image path as it appears in content.json (e.g., "images/fig_2.jpg")
    target_img_path = f"images/{json_file_path.stem}.jpg"

    with open(content_json_path, "r", encoding="utf-8") as f:
        content_data = json.load(f)

    img_index = -1
    caption_text = ""

    # Locate the image block in the flat JSON array
    for idx, block in enumerate(content_data):
        if block.get("type") == "image" and block.get("img_path") == target_img_path:
            img_index = idx
            if "img_caption" in block and block["img_caption"]:
                caption_text = " ".join(block["img_caption"])
            break

    if img_index == -1:
        return "Specific context not found for this image."

    # Gather text blocks BEFORE the image
    text_before = []
    for i in range(img_index - 1, -1, -1):
        block = content_data[i]
        if block.get("type") == "text" and "text" in block:
            text_before.insert(0, block["text"])  # Keep chronological order
            if len(text_before) == window_size:
                break

    # Gather text blocks AFTER the image
    text_after = []
    for i in range(img_index + 1, len(content_data)):
        block = content_data[i]
        if block.get("type") == "text" and "text" in block:
            text_after.append(block["text"])
            if len(text_after) == window_size:
                break

    # Assemble the final context string
    context_blocks = []
    if caption_text:
        context_blocks.append(f"Image Caption: {caption_text}")

    context_blocks.extend(text_before)
    context_blocks.extend(text_after)

    return "\n\n".join(context_blocks)

In [ ]:
def generate_smt(image, instruction, generator, max_new_tokens=256):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": instruction},
            ],
        }
    ]

    prompt = Chat(messages)

    return generator(prompt, max_new_tokens=max_new_tokens, do_sample=False)


def validate_smt(code):
    """
    Validates SMT-LIB code by executing the cvc5 binary.
    Returns (success, output_or_error).
    """
    # Create a temporary file for the SMT-LIB script
    with tempfile.NamedTemporaryFile(mode="w", suffix=".smt2", delete=False) as tf:
        tf.write(code)
        temp_path = tf.name

    try:
        # Run cvc5 with model production enabled
        result = subprocess.run(
            [CVC5_PATH, "--lang", "smt2", "--produce-models", temp_path],
            capture_output=True,
            text=True,
            timeout=15,  # Safety timeout
        )

        if result.returncode == 0:
            # Check if it's satisfiable or has errors in output
            output = result.stdout.strip()
            if "error" in output.lower():
                return False, f"cvc5 Logic Error:\n{output}"
            return True, output
        else:
            return False, f"cvc5 Syntax/Execution Error:\n{result.stderr}"

    except subprocess.TimeoutExpired:
        return (
            False,
            "cvc5 Error: Solver timed out (potential infinite loop or complex search).",
        )
    except Exception as e:
        return False, f"Internal Execution Error: {str(e)}"
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)


def reflect(image_data, initial_instruction, generator, max_retries=3):
    """
    Uses generate_smt to generate SMT-LIB code, then reflects on cvc5 output
    to fix logical errors over multiple turns.
    """
    # Current instruction starts as the initial prompt
    # In later turns, this string will grow to include the conversation history
    current_instruction = initial_instruction

    for _ in range(max_retries):
        # 1. Get guided output from the model
        raw_smt_code = generate_smt(
            image_data, current_instruction, generator, MAX_NEW_TOKENS
        )

        # 2. Validate using the cvc5 subprocess
        success, result = validate_smt(raw_smt_code)

        # Check for logical success (cvc5 returns 'sat')
        if success and "unsat" not in result.lower():
            return raw_smt_code, result

        # 3. If failed, construct the reflection prompt for the next turn
        # We append the failure and the request for correction to the instruction string
        error_context = (
            "The solver returned UNSAT (Logical contradiction)" if success else result
        )

        reflection_turn = (
            f"\n\nAssistant produced:\n{raw_smt_code}\n\n"
            f"Feedback from Solver:\n{error_context}\n\n"
            f"Please identify the incorrect spatial constraint and provide a corrected SMT-LIB script."
        )

        # Update the instruction for the next call to query_qwen
        current_instruction += reflection_turn

    return None, "Failed to reach a valid logical formulation."

In [ ]:
if STATE_FILE.exists():
    with open(STATE_FILE, "r") as f:
        saved_state = json.load(f)
    state = defaultdict(lambda: defaultdict(dict), saved_state)
    print(f"Loaded existing state from {STATE_FILE}. Resuming inference...")
else:
    state = defaultdict(lambda: defaultdict(dict))

In [ ]:
json_files = list(CASE_DIR.rglob("*.json"))
pbar = tqdm(json_files, desc="Generating SMT-LIB")

for json_file in pbar:
    fullpath = str(json_file)
    if "content.json" in fullpath or "images" not in fullpath or ".vscode" in fullpath:
        continue

    with open(json_file, "r") as f:
        data = json.load(f)

    sample_id = data.get("sample_id", json_file.stem)
    img_path = json_file.with_suffix(".jpg")
    if not img_path.exists():
        continue

    # Load state (ensure sample_id exists in dict)
    if sample_id not in state:
        state[sample_id] = {}

    full_img = Image.open(img_path.absolute())
    context = get_paper_context(json_file)
    bboxes = data.get("bbox", {})

    for sub_key, q_list in data.get("vqa", {}).items():
        if sub_key not in bboxes:
            continue

        if sub_key not in state[sample_id]:
            state[sample_id][sub_key] = {}

        # Crop logic
        box = bboxes[sub_key]
        sub_image = full_img.crop(
            (box["x"], box["y"], box["x"] + box["width"], box["y"] + box["height"])
        )

        for q_obj in q_list:
            question_text = q_obj.get("question") or q_obj.get("questions")
            question_text = q_obj.get("question") or q_obj.get("questions")
            question_type = q_obj.get("question_type", "")
            answer_type = q_obj.get("answer_type", "")

            # Check if already processed
            if question_text in state[sample_id][sub_key]:
                continue

            pbar.set_description(f"Processing {sample_id} - {sub_key}")

            instruction = PROMPTS[answer_type].format(
                question=question_text,
                question_type=question_type,
                answer_type=answer_type,
                context=context,
            )

            # We pass the 'llm' wrapper, not the raw 'model'
            fol_logic, solver_res = reflect(
                sub_image, instruction, generator, max_retries=3
            )

            # 3. PERSISTENCE
            if fol_logic:
                state[sample_id][sub_key][question_text] = {
                    "code": fol_logic,
                    "solver_output": solver_res,
                }
            else:
                state[sample_id][sub_key][question_text] = "FAILED"

            # Save state immediately to handle interruptions
            with open(STATE_FILE, "w") as f:
                json.dump(state, f, indent=2)

In [ ]:
with open(SMT_FILE, "w") as f:
    json.dump(state, f, indent=2)

print(f"Saved {len(state)} samples to {SMT_FILE}")

if STATE_FILE.exists():
    STATE_FILE.unlink()